<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/08_tab_gen_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0 - Best Practice)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# -----------------------------------------------------------------
# 1. Google Drive 마운트
# -----------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# -----------------------------------------------------------------
# 2. GitHub 최신화 및 경로 설정 (충돌 없는 강제 동기화)
# -----------------------------------------------------------------
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

# 핵심: 작업 디렉토리를 프로젝트 루트로 완벽히 고정하여 requirements.txt를 찾게 함
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# -----------------------------------------------------------------
# 3. 커스텀 모듈(src) 실행을 통한 의존성 설치 및 데이터셋 로드
# -----------------------------------------------------------------
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # env_setup.py 내부의 함수를 호출하여 requirements.txt 기반 설치 실행
    # (이 단계에서 torchcrepe, pretty_midi 등이 정상 설치됩니다)
    init_colab_env()

    # 데이터셋 복사
    MY_DRIVE_DATA_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_DATA_PATH, force_update=False)

except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")
    print("   (src/utils.py의 'import shutil' 오타가 수정되었는지 확인하세요!)")
except Exception as e:
    print(f"❌ 셋업 중단: {e}")

# -----------------------------------------------------------------
# 4. 글로벌 라이브러리 사전 적재
# -----------------------------------------------------------------
import librosa
import numpy as np
import pandas as pd
import torchcrepe
import pretty_midi

print("\n🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.")

🚀 Bass Separator 통합 환경 설정을 시작합니다...
🔄 레포지토리 최신화 중... (Git Fetch & Reset)
🚀 환경 설정을 시작합니다...

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 확인 중...
📄 requirements.txt 파일을 발견했습니다. 의존성 패키지를 설치합니다...
📦 패키지 일괄 설치 진행 중...
✅ 패키지 일괄 설치 완료.

🏥 설치 무결성 점검 (Health Check)...
✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!

🎉 모든 환경 설정이 완료되었습니다!
✅ 데이터가 이미 준비되어 있습니다: ./dataset
   (업데이트하려면 force_update=True 옵션을 사용하세요)

🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.


In [2]:
class BassTabGenerator:
    def __init__(self, sr=16000, hop_length=160):
        # 4현 베이스 표준 튜닝 (E1, A1, D2, G2)
        self.tuning = [28, 33, 38, 43]
        self.string_names = ["E", "A", "D", "G"]

        # Phase 4 파이프라인과 완벽히 동기화된 해상도
        self.sr = sr
        self.hop_length = hop_length
        self.events = []

    def get_fret_candidates(self, hz):
        """
        [확장성 설계] 주파수를 받아 가능한 '모든' 운지 위치 반환.
        향후 Phase 5의 Viterbi HMM 모델이 이 후보군들을 상태(State) 공간으로 활용합니다.
        """
        if hz is None or hz == 0 or np.isnan(hz):
            return []

        midi_note = int(round(librosa.hz_to_midi(hz)))
        candidates = []

        for string_idx, open_note in enumerate(self.tuning):
            fret = midi_note - open_note
            if 0 <= fret <= 24:
                candidates.append((string_idx, fret))

        return candidates

    def choose_fret_greedy(self, candidates):
        """
        [임시 로직] 가장 낮은 프렛을 우선 선택 (Lowest Fret Priority)
        Viterbi 알고리즘 개발 전까지 사용할 베이스라인 디코더입니다.
        """
        if not candidates:
            return None
        return min(candidates, key=lambda x: x[1])

    def parse_f0_to_events(self, f0_array):
        """
        [성능 개선] 파형 재분석 폐기.
        Phase 4에서 정제 완료된 f0_array의 결측치(NaN)와 변화량만 추적하여 Onset을 역산합니다.
        """
        self.events = []
        frame_time = self.hop_length / self.sr
        current_note = None

        for i, hz in enumerate(f0_array):
            is_valid = (not np.isnan(hz)) and (hz > 0)
            midi_note = int(round(librosa.hz_to_midi(hz))) if is_valid else None

            # 새로운 노트 시작점 감지 로직:
            # 1. 유효한 음이며
            # 2. 이전 프레임과 음높이가 다르거나(해머링/슬라이드), 직전이 휴지부(None)였을 때
            if is_valid and midi_note != current_note:
                candidates = self.get_fret_candidates(hz)
                pos = self.choose_fret_greedy(candidates)

                if pos:
                    self.events.append({
                        'time': i * frame_time,
                        'string_idx': pos[0],
                        'fret': pos[1],
                        'midi_note': midi_note
                    })
                current_note = midi_note

            # 음이 끊기면 현재 노트 초기화
            elif not is_valid:
                current_note = None

    def display_tab(self, chars_per_line=80):
        """
        [UI 안정화] 긴 휴지기(Rest)에 의한 무한 대시(-) 출력 및 줄바꿈 버그 수정
        """
        if not self.events:
            print("⚠️ 시각화할 노트 이벤트가 없습니다.")
            return

        print("\n🎸 Generated Bass Tab (Standard Tuning G-D-A-E)\n")
        line_buffers = ["G |", "D |", "A |", "E |"]
        last_time = 0.0

        for event in self.events:
            string_idx = event['string_idx']
            fret = event['fret']

            # 리듬 간격 계산 (최소 2칸, 최대 12칸으로 상한선 설정하여 악보 찢어짐 방지)
            time_diff = event['time'] - last_time
            num_dashes = max(2, min(12, int(time_diff * 10)))
            spacer = "-" * num_dashes

            fret_str = str(fret)
            added_length = len(spacer) + len(fret_str)

            # 콘솔 가로폭 한계 도달 시 사전 줄바꿈
            if len(line_buffers[0]) + added_length > chars_per_line:
                self._print_system(line_buffers)
                line_buffers = ["G |", "D |", "A |", "E |"]
                spacer = "-" * 2  # 새 줄은 간격 초기화

            # 4개 현 버퍼 채우기
            for i in range(4):
                current_string_target = 3 - i
                if current_string_target == string_idx:
                    line_buffers[i] += spacer + fret_str
                else:
                    line_buffers[i] += spacer + ("-" * len(fret_str))

            last_time = event['time']

        # 남은 버퍼 최종 출력
        if len(line_buffers[0]) > 3:
            self._print_system(line_buffers)

    def _print_system(self, buffers):
        for line in buffers:
            print(line + "-|")
        print("")

In [6]:
from src.bass_transcription import get_f0_crepe_robust

# 1. 경로 설정 및 방어적 검사 (Defensive Programming)
audio_path = '/content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav'
if not os.path.exists(audio_path):
    raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다. 경로를 확인하세요: {audio_path}")

print(f"📂 오디오 로드 중: {os.path.basename(audio_path)}")
y, sr = librosa.load(audio_path, sr=16000)

# 2. 피치 트래킹 (명시적 파라미터 제어)
print("🚀 피치 트래킹 실행 중 (Tiny 모델로 고속 추론)...")
# 방금 최적화한 model_capacity='tiny'를 명시적으로 전달하여 속도 확보
f0_data = get_f0_crepe_robust(y, sr, hop_length=160, model_capacity='tiny', batch_size=512)

# 3. 타브 악보 생성
print("📝 타브 악보 렌더링 중...")
tab_gen = BassTabGenerator(sr=16000, hop_length=160)
tab_gen.parse_f0_to_events(f0_data)

# 4. 결과 출력
tab_gen.display_tab(chars_per_line=80)
print("🎉 테스트 완료.")

# 5. 추출된 이벤트 데이터 일부 확인 (디버깅용)
print("\n🔍 [추출된 이벤트 데이터 Sample (Top 5)]")
for idx, event in enumerate(tab_gen.events[:5]):
    print(f" - Note {idx+1}: Time={event['time']:.2f}s, String={tab_gen.string_names[event['string_idx']]}, Fret={event['fret']}, MIDI={event['midi_note']}")

# 전체 이벤트 배열 반환
tab_gen.events

📂 오디오 로드 중: performance_test_demo(bass).wav
🚀 피치 트래킹 실행 중 (Tiny 모델로 고속 추론)...
⚠️ 경고: GPU가 감지되지 않아 연산이 매우 느려질 수 있습니다.
📝 타브 악보 렌더링 중...

🎸 Generated Bass Tab (Standard Tuning G-D-A-E)

G |----------------------------------------------------------------------------|
D |--0---------0--0------------------------------------------------------------|
A |---------------------------------------------3----3------------0----0-----0-|
E |--------------------3--------3--3--------------------------------------4----|

G |---------------------------------------------------------------------------|
D |----------0---------0--0---------------------------------------------------|
A |--0----0----------------------------------------------3----3-----3-------0-|
E |----------------------------3-------4--3--3--------------------------------|

G |--------------------------------------------------------------------|
D |---------------------0----0----------0-----------0--3----3--0-------|
A |--0--0--0------3--4---

[{'time': 0.08, 'string_idx': 2, 'fret': 0, 'midi_note': 38},
 {'time': 1.04, 'string_idx': 2, 'fret': 0, 'midi_note': 38},
 {'time': 1.32, 'string_idx': 2, 'fret': 0, 'midi_note': 38},
 {'time': 1.74, 'string_idx': 0, 'fret': 3, 'midi_note': 31},
 {'time': 2.59, 'string_idx': 0, 'fret': 3, 'midi_note': 31},
 {'time': 2.69, 'string_idx': 0, 'fret': 3, 'midi_note': 31},
 {'time': 3.92, 'string_idx': 1, 'fret': 3, 'midi_note': 36},
 {'time': 4.38, 'string_idx': 1, 'fret': 3, 'midi_note': 36},
 {'time': 5.59, 'string_idx': 1, 'fret': 0, 'midi_note': 33},
 {'time': 6.05, 'string_idx': 1, 'fret': 0, 'midi_note': 33},
 {'time': 6.3100000000000005, 'string_idx': 0, 'fret': 4, 'midi_note': 32},
 {'time': 6.5200000000000005, 'string_idx': 1, 'fret': 0, 'midi_note': 33},
 {'time': 7.0200000000000005, 'string_idx': 1, 'fret': 0, 'midi_note': 33},
 {'time': 7.48, 'string_idx': 1, 'fret': 0, 'midi_note': 33},
 {'time': 7.7, 'string_idx': 2, 'fret': 0, 'midi_note': 38},
 {'time': 8.69, 'string_idx':